# Domain Adaptation Dataset Pipeline

Pipeline untuk menghasilkan dataset domain adaptation dari materi kursus Python Basics.

**Tahap 1** dari pendekatan hibrida:
- **Tahap 1 (notebook ini)**: Domain Adaptation → `IndoT5-Python`
- **Tahap 2**: Task-Specific AQG → `IndoT5-Python-AQG`

**Format yang dihasilkan:**
| Format | Deskripsi | LLM? |
|--------|-----------|------|
| `span_corruption` | Masking 15% token gaya T5 pre-training | ❌ |
| `qa_generic` | Ekstrak QA dari bold/inline code/heading | ❌ |
| `summarization` | Ringkasan 2-4 kalimat via LLM | ✅ |

---

## 0. Setup

In [1]:
import sys
import os
import json
import shutil
from pathlib import Path
from collections import Counter

# Pastikan root project ada di sys.path
project_root = Path().resolve().parent.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Load .env
from dotenv import load_dotenv
load_dotenv(project_root / '.env')

print(f"Project root : {project_root}")
print(f"Python       : {sys.version.split()[0]}")

Project root : D:\2-Project\AQG
Python       : 3.12.13


## 1. Konfigurasi Pipeline

In [2]:
MATERI_DIR  = str(project_root / 'dataset_aqg' / 'materi')
OUTPUT_DIR  = str(project_root / 'dataset_aqg' / 'output_domain')

# Format yang akan dihasilkan
# Untuk zero-cost run (tanpa LLM): ['span_corruption', 'qa_generic']
# Untuk full run (dengan LLM)    : ['span_corruption', 'qa_generic', 'summarization']
FORMATS = ['span_corruption', 'qa_generic']

MAX_PER_CHUNK = 3
LLM_PROVIDER = None  # atau 'openrouter'

# Semua modul yang tersedia
materi_path = Path(MATERI_DIR)
ALL_MODULES = sorted([p.name for p in materi_path.iterdir() if p.is_dir()])

print(f"Materi dir  : {MATERI_DIR}")
print(f"Output dir  : {OUTPUT_DIR}")
print(f"Formats     : {FORMATS}")
print(f"\nSemua modul ({len(ALL_MODULES)}):")
for m in ALL_MODULES:
    print(f"  {m}")

Materi dir  : D:\2-Project\AQG\dataset_aqg\materi
Output dir  : D:\2-Project\AQG\dataset_aqg\output_domain
Formats     : ['span_corruption', 'qa_generic']

Semua modul (11):
  01-Berkenalan-dengan-python
  02-berinteraksi-dengan-data
  03-ekspresi
  04-aksi-sekuensial
  05-control-flow
  06-array
  07-matriks
  08-subprogram
  09-oop
  10-style-guide
  11-unit-testing


## 2. Preview Chunker

In [3]:
from src.dataset.step2.chunker import chunk_markdown

sample_file = str(project_root / 'dataset_aqg/materi/02-berinteraksi-dengan-data/03-type-data.md')
chunks = chunk_markdown(sample_file, max_tokens=512, min_tokens=128)

print(f"Total chunks: {len(chunks)}")
print(f"\n{'─'*60}")
for i, c in enumerate(chunks[:3]):
    print(f"\nChunk {i+1}: [{c.section_heading}] | tokens={c.token_count} | has_code={c.has_code}")
    print(c.text[:300] + ('...' if len(c.text) > 300 else ''))
    print('─'*60)

Total chunks: 6

────────────────────────────────────────────────────────────

Chunk 1: [# Tipe Data] | tokens=283 | has_code=False
# Tipe Data

Sebagaimana yang telah dijelaskan, setiap nilai yang digunakan dalam variabel adalah sebuah data. Data memiliki tipe yang berbeda-beda dan dapat kita temui dalam kehidupan sehari-hari. Simak kisah berikut. > "Seorang pria berumur 30 tahun menjalani kehidupan di ibu kota Jakarta. Pria te...
────────────────────────────────────────────────────────────

Chunk 2: [### Numbers] | tokens=335 | has_code=True
### Numbers

Tipe data `numbers` adalah tipe data angka yang terdiri dari tiga jenis:

| Jenis | Deskripsi | Contoh |
|-------|-----------|--------|
| `int` | Bilangan bulat positif atau negatif, tanpa desimal | `1`, `-20`, `999`, `0` |
| `float` | Bilangan riil, dapat berupa bilangan bulat atau des...
────────────────────────────────────────────────────────────

Chunk 3: [### String] | tokens=306 | has_code=True
### String

`String` merupakan ka

## 3. Preview Formatter

In [4]:
from src.dataset.step1.formatter import corrupt_spans, extract_qa_pairs

# Span corruption — chunk tanpa code block
sample_chunk = next((c for c in chunks if not c.has_code and c.token_count >= 30), chunks[0])

print("=" * 60)
print("SPAN CORRUPTION")
print("=" * 60)
sc_result = corrupt_spans(sample_chunk)
print(f"Input  : {sc_result.input[:200]}")
print(f"Target : {sc_result.target[:200]}")
print(f"Format : {sc_result.metadata['format']}")

SPAN CORRUPTION
Input  : # <extra_id_0> telah dijelaskan, setiap nilai yang digunakan dalam variabel adalah sebuah data. Data memiliki tipe yang berbeda-beda dan dapat kita temui dalam kehidupan sehari-hari. Simak kisah berik
Target : <extra_id_0> Tipe Data Sebagaimana yang <extra_id_1> yang dapat diambil dari <extra_id_2> - **Umur** — <extra_id_3> *numbers* dengan <extra_id_4> 50 huruf. - <extra_id_5> primitif** dan **tipe <extra_
Format : span_corruption


In [5]:
print("=" * 60)
print("QA GENERIC")
print("=" * 60)

# Cari chunk dengan bold term atau inline code (bukan di blockquote)
qa_chunk = next(
    (c for c in chunks if '**' in c.text or '`' in c.text),
    chunks[0]
)
qa_results = extract_qa_pairs(qa_chunk)
print(f"QA pairs ditemukan: {len(qa_results)}")
for i, qa in enumerate(qa_results[:5]):
    print(f"\nQA {i+1}:")
    print(f"  Input  : {qa.input}")
    print(f"  Target : {qa.target[:150]}")

QA GENERIC
QA pairs ditemukan: 2

QA 1:
  Input  : Apa itu tipe data primitif dalam Python?
  Target : Dalam Python, tipe data dikelompokkan menjadi dua: **tipe data primitif** dan **tipe data collection**.

QA 2:
  Input  : Apa itu tipe data collection dalam Python?
  Target : Dalam Python, tipe data dikelompokkan menjadi dua: **tipe data primitif** dan **tipe data collection**.


## 4. Jalankan Pipeline — Preview (2 Modul, Zero Cost)

Gunakan `module_filter` untuk membatasi modul tanpa perlu copy folder.

In [6]:
from src.pipeline.domain_pipeline import run_domain_pipeline

output_preview = str(project_root / 'dataset_aqg' / 'output_domain_preview')

# Reset output preview
if Path(output_preview).exists():
    shutil.rmtree(output_preview)

# Pilih 2 modul pertama — tidak perlu copy folder
PREVIEW_MODULES = ALL_MODULES[:2]
print(f"Preview modul: {PREVIEW_MODULES}")

summary = run_domain_pipeline(
    materi_dir=MATERI_DIR,
    output_dir=output_preview,
    formats=['span_corruption', 'qa_generic'],
    max_per_chunk=3,
    llm_provider=None,
    write_final=True,
    module_filter=PREVIEW_MODULES,  # ← filter langsung, tanpa copy
)

print("\nSummary:", summary)

Preview modul: ['01-Berkenalan-dengan-python', '02-berinteraksi-dengan-data']

Domain Adaptation Pipeline
Formats  : ['span_corruption', 'qa_generic']
Modul    : 2 (filtered)
Output   : D:\2-Project\AQG\dataset_aqg\output_domain_preview


[MODULE] 01-Berkenalan-dengan-python
  Chunks  : 16 dari 7 file


  Raw     : 47 data points
  Valid   : 47 | Failed: 0

[MODULE] 02-berinteraksi-dengan-data
  Chunks  : 15 dari 6 file


  Raw     : 44 data points
  Valid   : 43 | Failed: 1

[WRITE] Menulis final splits...
[DONE] 90 data points → D:\2-Project\AQG\dataset_aqg\output_domain_preview
       Train: 72 | Val: 9 | Test: 9

[SUMMARY]
  Raw generated : 91
  Valid         : 90
  Failed        : 1
  span_corruption     : 30
  qa_generic          : 60

Summary: {'total_raw': 91, 'total_valid': 90, 'total_failed': 1, 'format_counts': {'span_corruption': 30, 'qa_generic': 60}, 'output_dir': 'D:\\2-Project\\AQG\\dataset_aqg\\output_domain_preview'}


## 5. Verifikasi Output JSONL

In [7]:
output_path = Path(output_preview)

print("File output:")
for f in sorted(output_path.iterdir()):
    size = f.stat().st_size if f.is_file() else '-'
    print(f"  {f.name:30s} {size} bytes")

File output:
  accumulated.jsonl              94725 bytes
  checkpoint.json                106 bytes
  dataset_info.json              325 bytes
  test.jsonl                     9901 bytes
  train.jsonl                    74181 bytes
  validation.jsonl               10643 bytes
  validation_failures.jsonl      835 bytes


In [8]:
# Load dan verifikasi train.jsonl
train_file = output_path / 'train.jsonl'
if train_file.exists():
    records = []
    with open(train_file, encoding='utf-8') as f:
        for line in f:
            records.append(json.loads(line.strip()))

    print(f"Train records: {len(records)}")
    print(f"\nContoh record pertama:")
    r = records[0]
    print(f"  Keys    : {list(r.keys())}")
    print(f"  Format  : {r['metadata']['format']}")
    print(f"  Input   : {r['input'][:150]}...")
    print(f"  Target  : {r['target'][:150]}...")
else:
    print("train.jsonl belum ada")

Train records: 72

Contoh record pertama:
  Keys    : ['input', 'target', 'metadata']
  Format  : qa_generic
  Input   : Apa itu membuat objek baru dalam Python?...
  Target  : Perhatikan contoh berikut:

Ketika Anda melakukan inisialisasi ulang variabel, Python sebenarnya **membuat objek baru** dengan nilai baru — bukan meng...


In [9]:
# Distribusi format di semua splits
all_records = []
for split in ['train', 'validation', 'test']:
    f = output_path / f'{split}.jsonl'
    if f.exists():
        with open(f, encoding='utf-8') as fp:
            for line in fp:
                all_records.append(json.loads(line.strip()))

format_dist = Counter(r['metadata']['format'] for r in all_records)
print(f"Total records: {len(all_records)}")
print(f"Format distribution:")
for fmt, cnt in format_dist.items():
    print(f"  {fmt:25s}: {cnt}")

Total records: 90
Format distribution:
  qa_generic               : 60
  span_corruption          : 30


In [10]:
# dataset_info.json
info_file = output_path / 'dataset_info.json'
if info_file.exists():
    with open(info_file, encoding='utf-8') as f:
        info = json.load(f)
    print(json.dumps(info, indent=2, ensure_ascii=False))

{
  "total": 90,
  "splits": {
    "train": 72,
    "validation": 9,
    "test": 9
  },
  "format_distribution": {
    "qa_generic": 60,
    "span_corruption": 30
  },
  "module_distribution": {
    "01-berkenalan-dengan-python": 47,
    "02-berinteraksi-dengan-data": 43
  },
  "generated_at": "2026-04-07"
}


## 6. Load dengan HuggingFace datasets

In [11]:
try:
    from datasets import load_dataset

    data_files = {k: str(output_path / f'{k}.jsonl')
                  for k in ['train', 'validation', 'test']
                  if (output_path / f'{k}.jsonl').exists()}

    if data_files:
        dataset = load_dataset('json', data_files=data_files)
        print(dataset)
        print(f"\nContoh dari train split:")
        print(dataset['train'][0])
    else:
        print("Tidak ada file JSONL yang ditemukan")
except ImportError:
    print("datasets library tidak terinstall. Jalankan: pip install datasets")
except Exception as e:
    print(f"Error: {e}")

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input', 'target', 'metadata'],
        num_rows: 72
    })
    validation: Dataset({
        features: ['input', 'target', 'metadata'],
        num_rows: 9
    })
    test: Dataset({
        features: ['input', 'target', 'metadata'],
        num_rows: 9
    })
})

Contoh dari train split:
{'input': 'Apa itu membuat objek baru dalam Python?', 'target': 'Perhatikan contoh berikut:\n\nKetika Anda melakukan inisialisasi ulang variabel, Python sebenarnya **membuat objek baru** dengan nilai baru — bukan mengubah nilai yang sudah ada.', 'metadata': {'format': 'qa_generic', 'source_file': 'D:\\2-Project\\AQG\\dataset_aqg\\materi\\02-berinteraksi-dengan-data\\03-type-data.md', 'module_name': '02-berinteraksi-dengan-data', 'section_heading': '### Numbers', 'token_count': 335, 'has_code': True}}


## 7. Jalankan Pipeline Full (Semua 11 Modul)

In [13]:
# ⚠️  JALANKAN CELL INI UNTUK FULL RUN (semua 11 modul)
RUN_FULL = True  # Ubah ke True untuk menjalankan

if RUN_FULL:
    summary_full = run_domain_pipeline(
        materi_dir=MATERI_DIR,
        output_dir=OUTPUT_DIR,
        formats=['span_corruption', 'qa_generic'],
        max_per_chunk=3,
        llm_provider=None,
        write_final=True,
        module_filter=None,  # None = semua modul
    )
    print("\nFull run summary:", summary_full)
else:
    print("Set RUN_FULL = True untuk menjalankan pipeline penuh.")


Domain Adaptation Pipeline
Formats  : ['span_corruption', 'qa_generic']
Modul    : 11
Output   : D:\2-Project\AQG\dataset_aqg\output_domain


[MODULE] 01-Berkenalan-dengan-python
  Chunks  : 16 dari 7 file


  01-Berkenalan-dengan-python:   0%|          | 0/16 [00:00<?, ?chunk/s]

  Raw     : 47 data points
  Valid   : 47 | Failed: 0

[MODULE] 02-berinteraksi-dengan-data
  Chunks  : 15 dari 6 file


  Raw     : 44 data points
  Valid   : 43 | Failed: 1

[MODULE] 03-ekspresi
  Chunks  : 13 dari 4 file


  Raw     : 39 data points
  Valid   : 39 | Failed: 0

[MODULE] 04-aksi-sekuensial
  Chunks  : 7 dari 4 file


...

  Raw     : 32 data points
  Valid   : 32 | Failed: 0

[MODULE] 09-oop
  Chunks  : 9 dari 4 file


  Raw     : 26 data points
  Valid   : 26 | Failed: 0

[MODULE] 10-style-guide
  Chunks  : 15 dari 5 file


  Raw     : 30 data points
  Valid   : 30 | Failed: 0

[MODULE] 11-unit-testing
  Chunks  : 9 dari 3 file


  Raw     : 25 data points
  Valid   : 25 | Failed: 0

[WRITE] Menulis final splits...


[DONE] 340 data points → D:\2-Project\AQG\dataset_aqg\output_domain
       Train: 271 | Val: 33 | Test: 36

[SUMMARY]
  Raw generated : 341
  Valid         : 340
  Failed        : 1
  span_corruption     : 118
  qa_generic          : 222

Full run summary: {'total_raw': 341, 'total_valid': 340, 'total_failed': 1, 'format_counts': {'span_corruption': 118, 'qa_generic': 222}, 'output_dir': 'D:\\2-Project\\AQG\\dataset_aqg\\output_domain'}
